# Outliers Detection Techniques

**A comprehensive guide to identifying, analyzing, and handling outliers using the Titanic dataset.**

## Project Overview

This project demonstrates multiple outlier detection techniques on the Titanic dataset. Outliers can significantly impact statistical analyses and machine learning models. We cover visual methods (box plots, scatter plots), statistical methods (IQR, Z-score), and discuss when to remove, cap, or keep outliers.

## Learning Objectives

1. Understand what outliers are and why they matter
2. Apply IQR-based outlier detection
3. Apply Z-score and Modified Z-score methods
4. Use visual techniques (box plots, scatter plots) to identify outliers
5. Learn when to remove vs. keep outliers
6. Understand the impact of outliers on mean, median, and standard deviation

## Business / Research Problem

Outliers can arise from data entry errors, measurement issues, or genuine extreme values. Proper handling is critical for:
- **Accurate statistics:** Outliers can distort mean, variance, and correlation
- **Model performance:** Many ML algorithms are sensitive to outliers
- **Decision making:** False outliers lead to wrong conclusions

**Key question:** *How do we systematically identify outliers and decide whether to remove, transform, or keep them?*

## Why This Analysis Matters

Every real-world dataset contains outliers. Knowing how to handle them properly is a foundational data science skill that affects every downstream analysis and model.

## Dataset Overview

We use the Titanic dataset which has clear outliers in numerical columns like `fare`, `age`, and `sibsp`/`parch`.

| Feature | Type | Description |
|---------|------|-------------|
| `age` | Numeric | Passenger age |
| `fare` | Numeric | Ticket fare (has extreme outliers) |
| `sibsp` | Numeric | Siblings/spouses aboard |
| `parch` | Numeric | Parents/children aboard |
| `pclass` | Categorical | Passenger class (1/2/3) |

## Dataset Source & License Notes

- **Source:** [Kaggle - Titanic Dataset](https://www.kaggle.com/datasets/heptapod/titanic)
- **License:** Public Domain
- **Usage:** Educational purposes

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'heptapod/titanic'
IQR_MULTIPLIER = 1.5
ZSCORE_THRESHOLD = 3
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'Available files: {csv_files}')
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nBasic Statistics:')
df.describe()

## Data Cleaning

Minimal cleaning — we keep outliers for now since detecting them is our goal.

In [ ]:
# Standardize column names to lowercase
df.columns = df.columns.str.lower().str.strip()

# Fill age with median for analysis purposes
if 'age' in df.columns:
    df['age'] = df['age'].fillna(df['age'].median())

# Identify numerical columns for outlier analysis
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical columns for analysis: {num_cols}')

## Exploratory Data Analysis

Let's first see the distributions to visually identify potential outliers.

In [ ]:
# Summary stats highlighting outlier-prone columns
for col in ['age', 'fare', 'sibsp', 'parch']:
    if col in df.columns:
        print(f'\n=== {col.upper()} ===')
        print(f'Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f}')
        print(f'Std: {df[col].std():.2f}')
        print(f'Skewness: {df[col].skew():.2f}')
        print(f'Min: {df[col].min()}, Max: {df[col].max()}')

## Univariate Analysis

Box plots are the classic tool for spotting outliers. Points beyond the whiskers are potential outliers.

In [ ]:
plot_cols = [c for c in ['age', 'fare', 'sibsp', 'parch'] if c in df.columns]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for i, col in enumerate(plot_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='steelblue')
    axes[i].set_title(f'Box Plot: {col}')

plt.tight_layout()
plt.show()

In [ ]:
# Histograms with KDE to see distribution shape
fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for i, col in enumerate(plot_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='coral', edgecolor='white', density=True)
    df[col].dropna().plot.kde(ax=axes[i], color='navy')
    axes[i].set_title(f'Distribution: {col}')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

Outliers may be visible in 2D scatter plots even when they look normal in 1D.

In [ ]:
if 'age' in df.columns and 'fare' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(df['age'], df['fare'], alpha=0.5, s=20, c='steelblue')
    ax.set_xlabel('Age')
    ax.set_ylabel('Fare')
    ax.set_title('Age vs Fare — Spot the Outliers')
    plt.tight_layout()
    plt.show()

## Feature-Specific Insights

### Method 1: IQR (Interquartile Range) Method

The IQR method defines outliers as points below Q1 - 1.5*IQR or above Q3 + 1.5*IQR. This is what box plots use.

In [ ]:
def detect_iqr_outliers(series, multiplier=1.5):
    """Detect outliers using IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    outliers = (series < lower) | (series > upper)
    return outliers, lower, upper

print('=== IQR Outlier Detection ===')
for col in plot_cols:
    outliers, lower, upper = detect_iqr_outliers(df[col].dropna())
    n_outliers = outliers.sum()
    print(f'\n{col}:')
    print(f'  Bounds: [{lower:.2f}, {upper:.2f}]')
    print(f'  Outliers: {n_outliers} ({n_outliers/len(df)*100:.1f}%)')

### Method 2: Z-Score Method

The Z-score measures how many standard deviations a point is from the mean. Typically, |Z| > 3 is considered an outlier.

In [ ]:
def detect_zscore_outliers(series, threshold=3):
    """Detect outliers using Z-score method."""
    z_scores = np.abs(stats.zscore(series.dropna()))
    return z_scores > threshold

print('=== Z-Score Outlier Detection (threshold=3) ===')
for col in plot_cols:
    clean = df[col].dropna()
    outliers = detect_zscore_outliers(clean)
    n_outliers = outliers.sum()
    print(f'{col}: {n_outliers} outliers ({n_outliers/len(clean)*100:.1f}%)')

### Method 3: Modified Z-Score (MAD-based)

The modified Z-score uses the Median Absolute Deviation (MAD) instead of standard deviation, making it more robust to outliers themselves.

In [ ]:
def detect_mad_outliers(series, threshold=3.5):
    """Detect outliers using Modified Z-score (MAD method)."""
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        return pd.Series([False] * len(series))
    modified_z = 0.6745 * (series - median) / mad
    return np.abs(modified_z) > threshold

print('=== Modified Z-Score (MAD) Outlier Detection ===')
for col in plot_cols:
    clean = df[col].dropna()
    outliers = detect_mad_outliers(clean)
    n_outliers = outliers.sum()
    print(f'{col}: {n_outliers} outliers ({n_outliers/len(clean)*100:.1f}%)')

## Statistical Checks / Hypothesis Testing

Compare how outliers affect statistical measures.

In [ ]:
# Impact of outliers on statistics
target_col = 'fare' if 'fare' in df.columns else plot_cols[0]
clean = df[target_col].dropna()
iqr_mask, _, _ = detect_iqr_outliers(clean)
no_outliers = clean[~iqr_mask]

print(f'=== Impact of Outliers on {target_col.upper()} ===')
print(f'{"Metric":<20} {"With Outliers":>15} {"Without Outliers":>18}')
print('-' * 55)
print(f'{"Mean":<20} {clean.mean():>15.2f} {no_outliers.mean():>18.2f}')
print(f'{"Median":<20} {clean.median():>15.2f} {no_outliers.median():>18.2f}')
print(f'{"Std Dev":<20} {clean.std():>15.2f} {no_outliers.std():>18.2f}')
print(f'{"Skewness":<20} {clean.skew():>15.2f} {no_outliers.skew():>18.2f}')
print(f'{"Count":<20} {len(clean):>15} {len(no_outliers):>18}')

In [ ]:
# Shapiro-Wilk normality test
sample = clean.sample(min(500, len(clean)), random_state=RANDOM_SEED)
stat, p_val = stats.shapiro(sample)
print(f'Shapiro-Wilk test for {target_col}: W={stat:.4f}, p={p_val:.2e}')
print(f'Result: {"Normal" if p_val > 0.05 else "Not normal"} distribution')

## Visual Analysis

Side-by-side comparison: with and without outliers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# With outliers
axes[0, 0].hist(clean, bins=30, color='steelblue', edgecolor='white')
axes[0, 0].axvline(clean.mean(), color='red', linestyle='--', label=f'Mean: {clean.mean():.1f}')
axes[0, 0].axvline(clean.median(), color='orange', linestyle='--', label=f'Median: {clean.median():.1f}')
axes[0, 0].set_title(f'{target_col} — WITH Outliers')
axes[0, 0].legend()

# Without outliers
axes[0, 1].hist(no_outliers, bins=30, color='mediumseagreen', edgecolor='white')
axes[0, 1].axvline(no_outliers.mean(), color='red', linestyle='--', label=f'Mean: {no_outliers.mean():.1f}')
axes[0, 1].axvline(no_outliers.median(), color='orange', linestyle='--', label=f'Median: {no_outliers.median():.1f}')
axes[0, 1].set_title(f'{target_col} — WITHOUT Outliers')
axes[0, 1].legend()

# Box plots comparison  
axes[1, 0].boxplot(clean, vert=True)
axes[1, 0].set_title(f'{target_col} Box Plot — WITH Outliers')

axes[1, 1].boxplot(no_outliers, vert=True)
axes[1, 1].set_title(f'{target_col} Box Plot — WITHOUT Outliers')

plt.tight_layout()
plt.show()

In [ ]:
# Method comparison: how many outliers does each method detect?
results = []
for col in plot_cols:
    clean_col = df[col].dropna()
    iqr_out, _, _ = detect_iqr_outliers(clean_col)
    z_out = detect_zscore_outliers(clean_col)
    mad_out = detect_mad_outliers(clean_col)
    results.append({
        'Feature': col,
        'IQR': iqr_out.sum(),
        'Z-Score': z_out.sum(),
        'Modified Z': mad_out.sum(),
        'Total Points': len(clean_col)
    })

comparison = pd.DataFrame(results)
print('=== Outlier Detection Method Comparison ===')
print(comparison.to_string(index=False))

comparison.set_index('Feature')[['IQR', 'Z-Score', 'Modified Z']].plot(
    kind='bar', figsize=(10, 5), color=['steelblue', 'coral', 'mediumseagreen'])
plt.title('Outlier Count by Detection Method')
plt.ylabel('Number of Outliers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Key Findings

1. **Fare has the most extreme outliers** — a few passengers paid dramatically more than the rest
2. **Different methods detect different counts** — IQR is usually the most aggressive, Z-score the most conservative
3. **Outliers dramatically affect the mean** but barely affect the median, confirming the median's robustness
4. **Standard deviation is heavily inflated** by outliers, affecting any downstream confidence intervals
5. **Skewed distributions produce more IQR outliers** because the IQR method assumes symmetry
6. **Domain knowledge is essential** — high fares for first-class passengers are legitimate, not errors

## Limitations

1. **No ground truth:** We don't know which points are genuine errors vs. legitimate extreme values
2. **Univariate methods only:** These methods check one variable at a time — multivariate outliers may be missed
3. **Threshold sensitivity:** Results change significantly with IQR multiplier or Z-score threshold
4. **Distribution assumptions:** Z-score assumes normality; IQR assumes reasonable symmetry
5. **Small dataset effects:** With ~1,300 records, removing many outliers significantly reduces sample size

## Recommendations / Next Steps

1. **Use domain knowledge first** — understand what extreme values mean in context
2. **Try multiple methods** and compare before deciding on treatment
3. **Consider winsorization** (capping) instead of removal
4. **Log-transform** skewed features to reduce outlier impact without removing data

### How to Extend This Analysis
- Implement DBSCAN or Isolation Forest for multivariate outlier detection
- Compare model accuracy with/without outlier treatment
- Build an automated outlier report function

## Common Mistakes

1. **Blindly removing all IQR outliers:** Some outliers are real and informative
2. **Using Z-score on non-normal data:** Z-score assumes normality — for skewed data, use IQR or Modified Z-score
3. **Removing outliers before understanding them:** Always investigate first
4. **Applying outlier removal to the test set:** In ML, fit outlier thresholds on training data only
5. **Ignoring multivariate outliers:** A point can be normal in each dimension but extreme in combination

## Mini Challenge / Exercises

1. Implement Tukey's fences with different multipliers (1.5, 2.0, 3.0) and compare results
2. Apply log transformation to the `fare` column and rerun outlier detection — how many fewer outliers?
3. Create a function that flags rows with outliers in ANY numerical column
4. Implement winsorization at the 5th and 95th percentiles and compare statistics
5. Use `sklearn.ensemble.IsolationForest` for multivariate outlier detection

In [ ]:
# Space for exercise solutions
# Exercise 1: for mult in [1.5, 2.0, 3.0]: detect_iqr_outliers(df['fare'], mult)
# Exercise 2: np.log1p(df['fare'])
# Exercise 3: multi-column outlier flag
# Exercise 4: from scipy.stats import mstats; mstats.winsorize()
# Exercise 5: from sklearn.ensemble import IsolationForest

## Final Summary / Key Takeaways

- **No single method is best for all cases** — always try multiple approaches
- **IQR is the most commonly used** visual method (box plots), good for exploratory analysis
- **Z-score works well for normally distributed data** but fails on skewed distributions
- **Modified Z-score (MAD) is more robust** and recommended for non-normal data
- **Domain knowledge trumps statistics** — always ask whether an extreme value makes sense in context
- **Document your decisions** — future analysts need to know what was removed and why

The goal of outlier detection is not to remove data points, but to understand your data better and make informed decisions about its treatment.